In [1]:
import os
os.chdir('D:\\ai-engineering-buildcamp-codespace\\03-agents\\03-agents-framework\\')
print(os.getcwd())

D:\ai-engineering-buildcamp-codespace\03-agents\03-agents-framework


In [2]:
import pandas as pd
import requests
from pathlib import Path
from urllib.parse import urlparse
from markitdown import MarkItDown
from typing import Any, Dict, Iterable, List
import json
from pydantic import BaseModel, Field
from typing import Literal
from minsearch import AppendableIndex
from gitsource import GithubRepositoryDataReader, chunk_documents

In [4]:
from dotenv import find_dotenv, load_dotenv

load_dotenv(find_dotenv())

True

In [5]:
from openai import OpenAI
openai_client  = OpenAI()

In [6]:
from gitsource import GithubRepositoryDataReader
from minsearch import AppendableIndex

reader = GithubRepositoryDataReader(
    repo_owner="evidentlyai",
    repo_name="docs",
    allowed_extensions={"md", "mdx"},
)
files = reader.read()

parsed_docs = [doc.parse() for doc in files]

index = AppendableIndex(
    text_fields=["title", "description", "content"],
    keyword_fields=["filename"]
)
index.fit(parsed_docs)

In [7]:
from minsearch import Highlighter, Tokenizer
from minsearch.tokenizer import DEFAULT_ENGLISH_STOP_WORDS

In [8]:
stopwords = DEFAULT_ENGLISH_STOP_WORDS | {'evidently'}

highlighter = Highlighter(
    highlight_fields=['content'],
    max_matches=3,
    snippet_size=50,
    tokenizer=Tokenizer(stemmer='snowball', stop_words=stopwords)
)

In [9]:
file_index = {doc['filename']: doc['content'] for doc in parsed_docs}

In [10]:
from typing import Any, Dict, List


class SearchTools:
    """
    Provides search and file retrieval utilities over an indexed data store.
    """

    def __init__(
        self,
        index: Any,
        highlighter: Any,
        file_index: Dict[str, str]
    ) -> None:
        """
        Initialize the SearchTools instance.

        Args:
            index: Searchable index providing a `search` method.
            highlighter: Highlighter used to annotate search results.
            file_index (Dict[str, str]): Mapping of filenames to file contents.
        """
        self.index = index
        self.highlighter = highlighter
        self.file_index = file_index

    def search(self, query: str) -> List[Dict[str, Any]]:
        """
        Search the index for results matching a query and highlight them.

        Args:
            query (str): The search query to look up in the index.

        Returns:
            List[Dict[str, Any]]: A list of highlighted search result objects.
        """
        search_results = self.index.search(
            query=query,
            num_results=5
        )

        return self.highlighter.highlight(query, search_results)

    def get_file(self, filename: str) -> str:
        """
        Retrieve a file's contents by filename.

        Args:
            filename (str): The filename of the file to retrieve.

        Returns:
            str: The file contents if found, otherwise an error message.
        """
        if filename in self.file_index:
            return self.file_index[filename]
        return f"file {filename} does not exist"

In [11]:
search_tools = SearchTools(index, highlighter, file_index)

In [12]:
instructions = """
You're a documentation assistant.

Answer the user question using only the documentation knowledge base.

Make 3 iterations:

1) First iteration:
   - Perform one search using the search tool to identify potentially relevant documents.
   - Explain (in 2–3 sentences) why this search query is appropriate for the user question.

2) Second iteration:
   - Analyze the results from the previous search.
   - Based on the filenames or documents returned, perform:
       - Up to 2 additional search queries to refine or expand coverage, and
       - One or more get_file calls to retrieve the full content of the most relevant documents.
   - For each search or get_file call, explain (in 2–3 sentences) why this action is necessary and how it helps answer the question.

3) Third iteration:
   - Analyze the retrieved document contents from get_file.
   - Synthesize the information from these documents into a final answer to the user.

IMPORTANT:
- At every step, explicitly explain your reasoning for each search query or file retrieval.
- Use only facts found in the documentation knowledge base.
- Do not introduce outside knowledge or assumptions.
- If the answer cannot be found in the retrieved documents, clearly inform the user.

Additional notes:
- The knowledge base is entirely about Evidently, so you do not need to include the word "evidently" in search queries.
- Prefer retrieving and analyzing full documents (via get_file) before producing the final answer.
"""

In [13]:

 
from pydantic_ai import Agent
from toyaikit.tools import get_instance_methods

In [14]:
# tools = [search_tools.search, search_tools.get_file]
tools = get_instance_methods(search_tools)

In [15]:
search_agent = Agent(
    name='search',
    model='openai:gpt-4o-mini',
    instructions=instructions,
    tools=tools
)
    

In [16]:
 
query = 'how do I use evidently to monitor my machine learning models?'

In [17]:
result = await search_agent.run(query)

In [19]:
print(result.output)

After retrieving the documents titled "Overview" and "Batch monitoring," I've gathered useful information on how to monitor machine learning models using Evidently.

1. **Overview**: This document discusses the core function of AI observability, emphasizing the importance of evaluating the quality of inputs and outputs of AI applications in production. It introduces batch monitoring jobs as a way to run evaluations at regular intervals, leveraging the Evidently Python library to implement metric calculations and visualize results through dashboards. The benefits highlighted include control over evaluation scheduling, decoupling of log storage and monitoring metrics, and suitability for various ML evaluation scenarios.

2. **Batch Monitoring**: This document provides a practical outline for setting up batch evaluations, detailing how to configure an evaluation pipeline. It emphasizes the need to define reports with chosen metrics, run evaluations on a selected cadence (using tools like 

In [ ]:
for m in messages:
    print(m.kind)
    for p in m.parts:
        part_kind = p.part_kind
        if part_kind == 'user-prompt':
            print('USER:', p.content)
        if part_kind == 'tool-call':
            print('TOOL CALL:', p.tool_name, p.args)
        if part_kind == 'tool-return':
            print('TOOL RETURN:', p.tool_name)
        if part_kind == 'text':
            print(p.content)
    print()